# Denoising the Stanford Bunny

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch import optim
from tqdm.notebook import trange
import k3d
import sys
import os
import time
import copy
import trimesh
import mesh_to_sdf
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from models.model_architecture import BunnyNet
from training.optimizers import GaussNewtonWoodburyBig
from util.surface_sampling import sample_model_surface_binsearch, sample_model_surface_newton 
from util.error_metrics import chamfer_div, compute_distance 
from util.visualization.utils_mesh import get_mesh

torch.manual_seed(0)


device = 'cuda' #if torch.cuda.is_available() else 'cpu'
torch.set_default_device(device)

In [2]:
mesh = trimesh.load("bun_zipper.ply")
pts = torch.tensor(mesh.vertices, dtype=torch.float64)
fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(mesh.vertices, mesh.faces, color=0xbbbbbb, side='double', flat_shading=False)
fig.display()

/cluster/home/jaking/.local/lib/python3.11/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "float64" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/cluster/home/jaking/.local/lib/python3.11/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "int64" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


Output()

In [10]:
# bbox_min = mesh.bounds[0] - 0.01
# bbox_max = mesh.bounds[1] + 0.01
# num_samples = 10000

# samples = np.random.rand(num_samples, 3) * (bbox_max - bbox_min) + bbox_min

# # Compute signed distances (no bounding_box_padding here)
# sdf = mesh_to_sdf.mesh_to_sdf(
#     mesh,
#     samples,
#     surface_point_method='sample',  
#     sign_method='normal'            
# )

# x_train = torch.from_numpy(samples).double().to(device)
# y_train = torch.from_numpy(sdf).double().to(device)

# noise_level = 0.001  # Control the noise level here
# noisy_pts = pts + torch.randn_like(pts) * noise_level

# x_train = torch.cat([x_train, noisy_pts])
# y_train = torch.cat([y_train, torch.zeros(noisy_pts.shape[0], dtype=torch.float64)])

# # Input normalisation
# x_mean = x_train.mean(dim=0)
# x_std = x_train.std(dim=0)
# x_train = (x_train - x_mean) / x_std

# # Output normalisation
# y_scale = y_train.abs().max()
# y_train = y_train / y_scale


In [16]:
# Step 1: Generate random samples in bounding box
num_samples = 3000
bbox_min = mesh.bounds[0] - 0.01
bbox_max = mesh.bounds[1] + 0.01
samples = np.random.rand(num_samples, 3) * (bbox_max - bbox_min) + bbox_min

# Step 2: Compute signed distances for the generated points
sdf = mesh_to_sdf.mesh_to_sdf(
    mesh,
    samples,
    surface_point_method='sample',
    sign_method='normal'
)

x_train = torch.from_numpy(samples).double().to(device)
y_train = torch.from_numpy(sdf).double().to(device)

x_train = torch.cat([pts, x_train])
# y_train = torch.cat([torch.zeros(pts.shape[0], dtype=torch.float64), y_train])
# x_train = pts
# y_train = torch.zeros(pts.shape[0], dtype=torch.float64)

# Step 3: Normalize inputs
x_mean = x_train.mean(dim=0)
x_std = x_train.std(dim=0)
x_train = (x_train - x_mean) / x_std

# Step 4: Normalize outputs (SDF values)
y_scale = y_train.abs().max()
y_train = y_train / y_scale

pts_surface = x_train[:pts.shape[0]]
pts_off_surface = x_train[-num_samples:]

nearest_points, _, face_indices = mesh.nearest.on_surface(pts.cpu().numpy())
normals = torch.tensor(mesh.face_normals[face_indices])
normals = normals / x_std
normals = normals / (normals.norm(dim=1, keepdim=True)+1e-8)


# Step 5: Add noise to a subset of the surface points (after normalization)
# pts_normalized = x_train[-pts.shape[0]:]
# num_noisy_points = int(0.5 * pts_normalized.shape[0])  # Add noise to 10% of the surface points
# indices = torch.randperm(pts_normalized.shape[0])[:num_noisy_points]  # Randomly select a subset of points
# noise_level = 0.05
# noisy_pts = pts_normalized[indices] + torch.randn_like(pts_normalized[indices]) * noise_level  # Add noise only to selected points

# # Replace the noisy points in the dataset, without including the original ones
# x_train[-pts_normalized.shape[0]:][indices] = noisy_pts


In [47]:
# Step 1: Generate random samples in bounding box
num_samples = 3000
bbox_min = mesh.bounds[0] - 0.01
bbox_max = mesh.bounds[1] + 0.01
samples = np.random.rand(num_samples, 3) * (bbox_max - bbox_min) + bbox_min

# Step 2: Compute signed distances for the generated points
sdf = mesh_to_sdf.mesh_to_sdf(
    mesh,
    samples,
    surface_point_method='sample',
    sign_method='normal'
)

x_train = torch.from_numpy(samples).double().to(device)
y_train = torch.from_numpy(sdf).double().to(device)

x_train = torch.cat([x_train, pts])
y_train = torch.cat([y_train, torch.zeros(pts.shape[0], dtype=torch.float64)])

# Step 3: Normalize inputs
x_mean = x_train.mean(dim=0)
x_std = x_train.std(dim=0)
x_train = (x_train - x_mean) / x_std

# Step 4: Normalize outputs (SDF values)
y_scale = y_train.abs().max()
y_train = y_train / y_scale


# Step 5: Add noise to a subset of the surface points (after normalization)
# pts_normalized = x_train[-pts.shape[0]:]
# num_noisy_points = int(0.5 * pts_normalized.shape[0])  # Add noise to 10% of the surface points
# indices = torch.randperm(pts_normalized.shape[0])[:num_noisy_points]  # Randomly select a subset of points
# noise_level = 0.05
# noisy_pts = pts_normalized[indices] + torch.randn_like(pts_normalized[indices]) * noise_level  # Add noise only to selected points

# # Replace the noisy points in the dataset, without including the original ones
# x_train[-pts_normalized.shape[0]:][indices] = noisy_pts

# # Step 6: Concatenate noisy surface points and random samples
# x_train = torch.cat([x_train, noisy_pts], dim=0)

# # Step 7: The corresponding SDF values remain the same for the noisy surface points
# y_train = torch.cat([y_train, torch.zeros(noisy_pts.shape[0], dtype=torch.float64).to(device)], dim=0)


In [4]:
fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(mesh.vertices, mesh.faces, color=0xbbbbbb, side='double', flat_shading=False)
fig += k3d.points(x_train[-pts.shape[0]:].cpu().detach(), color=0x00ff00, point_size=0.005)
fig.display()

/cluster/home/jaking/.local/lib/python3.11/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "float64" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/cluster/home/jaking/.local/lib/python3.11/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "int64" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


Output()

In [40]:
model = BunnyNet(ks=[3, 128, 128, 128, 128, 128, 1])
model = model.double()

eikonal_weight = 1e-2
num_epochs = 10000
batch_size = 10000
lr = 1e-3

# Optimiser
optimizer = optim.Adam(model.parameters(), lr=lr)

# Supervised SDF loss
def sdf_loss(model, x, y):
    preds = model(x).squeeze(1)
    return 0.5 * (preds - y).square().mean()


# Training loop
for epoch in range(num_epochs):
    # Sample a batch
    indices = torch.randint(0, x_train.size(0), (batch_size,)).detach()
    x_batch = x_train[indices]
    y_batch = y_train[indices]
    indices_normals = torch.clamp(indices, max=normals.size(0) - 1)  # Ensure indices are valid for normals
    normals_batch = normals[indices_normals]
    x_batch_normals = x_train[indices_normals]

    optimizer.zero_grad()

    loss_data = sdf_loss(model, x_batch, y_batch)
    loss_eikonal = 0.5 * model.r_eikonal(model.params, x_batch).squeeze(1).square().mean()
    loss_normal = 0.5 * torch.norm(model.grad_x_f(model.params, x_batch_normals).squeeze(1) - normals_batch, dim=-1).square().mean()
    loss_total = loss_data + eikonal_weight*loss_eikonal + loss_normal

    loss_total.backward()
    optimizer.step()

    if epoch % 100 == 0 or epoch == num_epochs - 1:
        print(f"Epoch {epoch}: Total Loss = {loss_total.item():.6f}, Data = {loss_data.item():.6f}, Eikonal = {loss_eikonal.item():.6f}, Normal = {loss_normal.item():.6f}")

print("Pretraining completed!")

Epoch 0: Total Loss = 0.534445, Data = 0.016827, Eikonal = 0.359983, Normal = 0.514018
Epoch 100: Total Loss = 0.414532, Data = 0.006908, Eikonal = 0.197203, Normal = 0.405652
Epoch 200: Total Loss = 0.401498, Data = 0.006700, Eikonal = 0.189826, Normal = 0.392899
Epoch 300: Total Loss = 0.381516, Data = 0.005288, Eikonal = 0.176024, Normal = 0.374468
Epoch 400: Total Loss = 0.363441, Data = 0.005604, Eikonal = 0.163180, Normal = 0.356205
Epoch 500: Total Loss = 0.347239, Data = 0.004307, Eikonal = 0.149741, Normal = 0.341435
Epoch 600: Total Loss = 0.343095, Data = 0.004952, Eikonal = 0.139029, Normal = 0.336753
Epoch 700: Total Loss = 0.338677, Data = 0.005510, Eikonal = 0.136545, Normal = 0.331802
Epoch 800: Total Loss = 0.334567, Data = 0.005097, Eikonal = 0.136973, Normal = 0.328100
Epoch 900: Total Loss = 0.328875, Data = 0.004850, Eikonal = 0.134944, Normal = 0.322675
Epoch 1000: Total Loss = 0.333363, Data = 0.005269, Eikonal = 0.132365, Normal = 0.326771
Epoch 1100: Total Loss

In [17]:
model = BunnyNet(ks=[3, 32, 32, 32, 1])
model = model.double()

eikonal_weight = 1e-2
num_epochs = 10000
batch_size = 10000
batch_size_off_surface = 1000
lr = 1e-3

# Optimiser
optimizer = optim.Adam(model.parameters(), lr=lr)

# Supervised SDF loss
def sdf_loss(model, x, y=0):
    preds = model(x).squeeze(1)
    return 0.5 * (preds - y).square().mean()


# Training loop
for epoch in range(num_epochs):
    # Sample a batch
    # indices_surface = torch.randint(0, pts_surface.size(0), (batch_size_off_surface,)).detach()
    # pts_surface_batch = pts_surface[indices_surface]
    # y_surface_batch = torch.zeros(pts_surface_batch.shape[0])
    # # indices_normals = torch.clamp(indices, max=normals.size(0) - 1)  # Ensure indices are valid for normals
    # normals_batch = normals[indices_surface]
    # # x_batch_normals = x_train[indices_normals]
    # indices_off_surface = torch.randint(0, pts_off_surface.size(0), (batch_size,)).detach()
    # pts_off_surface_batch = pts_surface[indices_off_surface]
    # y_off_surface_batch = y_train[indices_off_surface]

    optimizer.zero_grad()
    loss_data = sdf_loss(model, pts_surface) + sdf_loss(model, pts_off_surface, y_train)
    loss_eikonal = 0.5 * model.r_eikonal(model.params, x_train).squeeze(1).square().mean()
    loss_normal = 0.5 * torch.norm(model.grad_x_f(model.params, pts_surface).squeeze(1) - normals, dim=-1).square().mean()
    loss_total = 2*loss_data + loss_normal + eikonal_weight*loss_eikonal

    loss_total.backward()
    optimizer.step()

    if epoch % 100 == 0 or epoch == num_epochs - 1:
        print(f"Epoch {epoch}: Total Loss = {loss_total.item():.6f}, Data = {loss_data.item():.6f}, Eikonal = {loss_eikonal.item():.6f}, Normal = {loss_normal.item():.6f}")
    if epoch % 1000 == 0 and epoch != 0:
        for param_group in optimizer.param_groups:
            param_group['lr'] *= 0.9

print("Pretraining completed!")

Epoch 0: Total Loss = 0.655799, Data = 0.061879, Eikonal = 0.338592, Normal = 0.528655
Epoch 100: Total Loss = 0.292230, Data = 0.017514, Eikonal = 0.155770, Normal = 0.255646
Epoch 200: Total Loss = 0.230188, Data = 0.016864, Eikonal = 0.100206, Normal = 0.195457
Epoch 300: Total Loss = 0.208282, Data = 0.014372, Eikonal = 0.092940, Normal = 0.178609
Epoch 400: Total Loss = 0.180716, Data = 0.012718, Eikonal = 0.077260, Normal = 0.154508
Epoch 500: Total Loss = 0.165881, Data = 0.012597, Eikonal = 0.064472, Normal = 0.140042
Epoch 600: Total Loss = 0.153918, Data = 0.012157, Eikonal = 0.058291, Normal = 0.129021
Epoch 700: Total Loss = 0.137067, Data = 0.011240, Eikonal = 0.050671, Normal = 0.114081
Epoch 800: Total Loss = 0.122186, Data = 0.009800, Eikonal = 0.044591, Normal = 0.102141
Epoch 900: Total Loss = 0.113716, Data = 0.009186, Eikonal = 0.040854, Normal = 0.094935
Epoch 1000: Total Loss = 0.107616, Data = 0.008786, Eikonal = 0.038511, Normal = 0.089658
Epoch 1100: Total Loss

In [29]:
verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=torch.tensor([-1.75, -1.6, -2.5], dtype=torch.float64),
    bbox_max=torch.tensor([2.4, 2.3, 1.7], dtype=torch.float64),
    chunks=2
)

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
# fig += k3d.points(x_train.cpu().detach(), color=0x00ff00, point_size=0.005)
# fig += k3d.points(pts_surface_true.cpu().detach(), point_size=0.05)
fig.display()

Output()

In [16]:
fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(mesh.vertices, mesh.faces, color=0xbbbbbb, side='double', flat_shading=False)
fig.display()

Output()